In [1]:
import random

class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)

        self.rows = rows
        self.cols = cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {
            "position": (r, c),
            "is_dirty": self.dirt[r][c]
        }

    def valid_moves(self):
        r, c = self.position
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.position

        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


In [3]:
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"

    return random.choice(env.valid_moves())

In [4]:
def run_agent(env, decide, max_steps=200):
    steps = 0

    for _ in range(max_steps):
        if env.is_clean():
            break

        percept = env.get_percept()
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1

    return steps


env = GridEnvironment(
    rows=4,
    cols=4,
    dirt_probability=0.5,
    seed=11
)

steps = run_agent(env, simple_reflex_agent)
print("Simple reflex (random) steps:", steps)

Simple reflex (random) steps: 48


In [5]:
import copy

In [7]:

def neighbor_position(pos, action):
    r, c = pos

    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c)
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        here = percept["position"]
        self.visited.add(here)

        if percept["is_dirty"]:
            return "Suck"

        moves = env.valid_moves()

        unvisited = [
            move for move in moves
            if neighbor_position(here, move) not in self.visited
        ]

        if unvisited:
            return random.choice(unvisited)

        return random.choice(moves)


In [8]:

random.seed(11)

env1 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
dirt_copy = copy.deepcopy(env1.dirt)

steps1 = run_agent(env1, simple_reflex_agent)
print("Simple reflex steps:", steps1)

random.seed(11)

env2 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
env2.dirt = copy.deepcopy(dirt_copy)

agent2 = ModelBasedAgent()
steps2 = run_agent(env2, agent2.act)

print("Model-based steps:", steps2)


Simple reflex steps: 48
Model-based steps: 42


In [10]:

def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal

    if r < gr:
        return "Down"
    if r > gr:
        return "Up"
    if c < gc:
        return "Right"
    if c > gc:
        return "Left"

    return "NoOp"


In [11]:

env = GridEnvironment(rows=4, cols=4)
env.position = (0, 0)

goal = (3, 2)

print("Start:", env.position, "Goal:", goal)

for step in range(20):
    if env.position == goal:
        print("Reached goal in", step, "steps.")
        break

    action = goal_based_agent(env.position, goal)
    env.apply_action(action)

    print(f"Step {step + 1}: moved {action} -> {env.position}")


Start: (0, 0) Goal: (3, 2)
Step 1: moved Down -> (1, 0)
Step 2: moved Down -> (2, 0)
Step 3: moved Down -> (3, 0)
Step 4: moved Right -> (3, 1)
Step 5: moved Right -> (3, 2)
Reached goal in 5 steps.


In [12]:

env.position = (1, 3)
goal = (0, 1)

print("Start:", env.position, "Goal:", goal)

for step in range(20):
    if env.position == goal:
        print("Reached goal in", step, "steps.")
        break

    action = goal_based_agent(env.position, goal)
    env.apply_action(action)

    print(f"Step {step + 1}: moved {action} -> {env.position}")


Start: (1, 3) Goal: (0, 1)
Step 1: moved Up -> (0, 3)
Step 2: moved Left -> (0, 2)
Step 3: moved Left -> (0, 1)
Reached goal in 3 steps.


In [27]:

def steps_per_layout(make_decide, env_class=GridEnvironment, layouts=200):
    results = []

    for i in range(layouts):
        env = env_class(
            rows=4,
            cols=4,
            dirt_probability=0.5,
            seed=i
        )

        steps = run_agent(env, make_decide())
        results.append(steps)

    return results


In [14]:

reflex_results = steps_per_layout(
    lambda: simple_reflex_agent
)

model_results = steps_per_layout(
    lambda: ModelBasedAgent().act
)

print("Reflex average:", sum(reflex_results) / len(reflex_results))
print("Model-based average:", sum(model_results) / len(model_results))

worse_count = sum(
    1 for a, b in zip(reflex_results, model_results)
    if b > a
)

print("Layouts where model-based took more steps:", worse_count)


Reflex average: 71.69
Model-based average: 29.23
Layouts where model-based took more steps: 5


In [29]:

env = SensorGridEnvironment(rows=4, cols=4, seed=11)

print("Percept:", env.get_percept())


Percept: {'position': (0, 0), 'is_dirty': True, 'dirty_neighbors': {'Right': False, 'Down': False}}


In [30]:

sensor_results = steps_per_layout(
    lambda: sensor_agent,
    env_class=SensorGridEnvironment
)

print("Sensor-based average:", sum(sensor_results) / len(sensor_results))


Sensor-based average: 30.235


In [32]:

class UtilityGridEnvironment(GridEnvironment):
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        super().__init__(rows, cols, dirt_probability, seed)

        self.values = [
            [random.randint(1, 5) for _ in range(cols)]
            for _ in range(rows)
        ]
        self.score = 0

    def clean_room(self):
        r, c = self.position

        if self.dirt[r][c]:
            self.score += self.values[r][c]
            self.dirt[r][c] = False


In [33]:

def nearest_room_agent(env):
    dirty_rooms = [
        (r, c)
        for r in range(env.rows)
        for c in range(env.cols)
        if env.dirt[r][c]
    ]

    if not dirty_rooms:
        return None

    here = env.position

    return min(
        dirty_rooms,
        key=lambda room: abs(room[0] - here[0]) + abs(room[1] - here[1])
    )


In [34]:

def utility_agent(env):
    dirty_rooms = [
        (r, c)
        for r in range(env.rows)
        for c in range(env.cols)
        if env.dirt[r][c]
    ]

    if not dirty_rooms:
        return None

    here = env.position

    return max(
        dirty_rooms,
        key=lambda room: env.values[room[0]][room[1]]
        / (abs(room[0] - here[0]) + abs(room[1] - here[1]) + 1)
    )


In [35]:

def run_scoring_agent(env, choose_target, max_steps=20):
    for _ in range(max_steps):
        if env.is_clean():
            break

        target = choose_target(env)

        if target is None:
            break

        if env.position == target:
            env.clean_room()
        else:
            action = goal_based_agent(env.position, target)
            env.apply_action(action)

    return env.score


nearest_scores = []
utility_scores = []

for i in range(200):
    env1 = UtilityGridEnvironment(seed=i)
    env2 = UtilityGridEnvironment(seed=i)

    # Ensure identical dirt and room values.
    env2.dirt = copy.deepcopy(env1.dirt)
    env2.values = copy.deepcopy(env1.values)

    nearest_scores.append(run_scoring_agent(env1, nearest_room_agent))
    utility_scores.append(run_scoring_agent(env2, utility_agent))

print("Nearest-room average:", sum(nearest_scores) / 200)
print("Utility average:", sum(utility_scores) / 200)

print("Layouts where utility scored higher:",
      sum(u > n for n, u in zip(nearest_scores, utility_scores)))


Nearest-room average: 22.04
Utility average: 22.08
Layouts where utility scored higher: 37
